# Module 6.1 — Testing & Test Automation

### Python Mastery · Track 6: Testing, Tooling & DevOps

Tests are how you know your code works, and keep knowing it works as you change it. This module covers the built-in `unittest` framework, mocking with `unittest.mock`, the popular `pytest` framework, and measuring how much of your code your tests exercise.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it (`Shift + Enter`).
- `unittest` and `mock` run directly in cells. `pytest` examples are written to a `.py` file and run with `!python -m pytest`, because pytest discovers and runs tests from files, exactly as in a real project.
- Attempt the exercises before consulting the solutions at the end.

### Learning objectives

After completing this module you will be able to:

1. Write tests with `unittest.TestCase` and its assertions.
2. Use `setUp` and `tearDown` for shared test fixtures.
3. Replace dependencies with `unittest.mock` (`Mock`, `patch`).
4. Write concise tests with `pytest`, including fixtures and parametrisation.
5. Measure test coverage and interpret the result.

**Prerequisites:** Tracks 1 to 5.

---

## Part 1 · `unittest`: Test Cases and Assertions

The standard library's `unittest` framework organises tests into classes that subclass `TestCase`. Each method whose name starts with `test` is a separate test. Inside, you use assertion methods such as `assertEqual` and `assertTrue` to state what you expect; if an assertion fails, the test fails with a clear report.

Notebooks have no command line to launch the test runner, so we run the suite programmatically with `unittest.main(..., exit=False)`, which works inside a cell.

In [ ]:
import unittest

# The code under test.
def add(a, b):
    return a + b

def is_even(n):
    return n % 2 == 0

# A test case groups related tests.
class TestMath(unittest.TestCase):
    def test_add_positive(self):
        self.assertEqual(add(2, 3), 5)          # expected == actual

    def test_add_negative(self):
        self.assertEqual(add(-1, -1), -2)

    def test_is_even_true(self):
        self.assertTrue(is_even(4))

    def test_is_even_false(self):
        self.assertFalse(is_even(5))

# Run the suite inside the notebook (argv and exit keep it from quitting the kernel).
unittest.main(argv=["ignored"], exit=False, verbosity=2)

`unittest` offers many assertions beyond equality. Common ones include `assertIn`, `assertRaises` (for expected exceptions), `assertAlmostEqual` (for floats), and `assertIsNone`. Using the most specific assertion gives the clearest failure messages.

In [ ]:
import unittest

def divide(a, b):
    if b == 0:
        raise ValueError("cannot divide by zero")
    return a / b

class TestDivide(unittest.TestCase):
    def test_result(self):
        self.assertAlmostEqual(divide(1, 3), 0.3333, places=3)   # float comparison

    def test_membership(self):
        self.assertIn("a", ["a", "b", "c"])

    def test_raises(self):
        # assertRaises checks that the expected exception is raised.
        with self.assertRaises(ValueError):
            divide(1, 0)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

## Part 2 · Fixtures with `setUp` and `tearDown`

When several tests need the same starting state, put the preparation in `setUp`, which runs **before each test**, and any cleanup in `tearDown`, which runs **after each test**. This keeps tests independent: each gets a fresh, known environment, so they do not interfere with one another.

In [ ]:
import unittest

class Stack:
    def __init__(self):
        self.items = []
    def push(self, x): self.items.append(x)
    def pop(self): return self.items.pop()
    def size(self): return len(self.items)

class TestStack(unittest.TestCase):
    def setUp(self):
        # Runs before EACH test, giving every test a fresh stack.
        self.stack = Stack()
        self.stack.push(1)
        self.stack.push(2)

    def test_size(self):
        self.assertEqual(self.stack.size(), 2)

    def test_pop_returns_last(self):
        self.assertEqual(self.stack.pop(), 2)
        self.assertEqual(self.stack.size(), 1)     # unaffected by other tests

    def tearDown(self):
        # Runs after EACH test; here just illustrative.
        self.stack = None

unittest.main(argv=["ignored"], exit=False, verbosity=2)

## Part 3 · Mocking with `unittest.mock`

A unit test should test one unit in isolation, not its dependencies (a network service, a database, the clock). A **mock** is a stand-in object that records how it was used and returns whatever you tell it to. `Mock` creates such an object; `patch` temporarily replaces a real name with a mock for the duration of a test.

In [ ]:
from unittest.mock import Mock

# A Mock records calls and returns configured values.
service = Mock()
service.fetch.return_value = {"status": "ok"}     # configure what fetch() returns

result = service.fetch("user-1")
print("returned:", result)

# The mock remembers how it was called, which we can assert on.
service.fetch.assert_called_once_with("user-1")
print("call count:", service.fetch.call_count)
print("call args:", service.fetch.call_args)

In [ ]:
import unittest
from unittest.mock import patch

# Code that depends on something external (here, a function we will replace).
def get_temperature():
    # Imagine this calls a weather API; we do not want a real call in tests.
    raise RuntimeError("real network call")

def describe_weather():
    temp = get_temperature()
    return "warm" if temp >= 25 else "cool"

class TestWeather(unittest.TestCase):
    @patch(f"{__name__}.get_temperature")          # replace get_temperature with a mock
    def test_warm(self, mock_temp):
        mock_temp.return_value = 30                 # control the dependency
        self.assertEqual(describe_weather(), "warm")

    @patch(f"{__name__}.get_temperature")
    def test_cool(self, mock_temp):
        mock_temp.return_value = 10
        self.assertEqual(describe_weather(), "cool")

unittest.main(argv=["ignored"], exit=False, verbosity=2)

## Part 4 · `pytest`: Less Boilerplate

`pytest` is the most popular third-party testing framework. It needs no classes: a test is any function named `test_*` that uses the plain `assert` statement, and pytest rewrites assertions to give rich failure messages. Because pytest discovers tests from files, the examples here are written to a `.py` file and run with `!python -m pytest`.

In [ ]:
%%writefile test_basics.py
# A pytest file: plain functions and plain assert statements.

def add(a, b):
    return a + b

def test_add():
    assert add(2, 3) == 5

def test_add_negative():
    assert add(-1, 1) == 0

def test_string_join():
    assert "-".join(["a", "b"]) == "a-b"

In [ ]:
# -q means quiet; pytest discovers and runs the test_* functions automatically.
!python -m pytest test_basics.py -q

`pytest` **fixtures** provide reusable setup through function arguments: define a function decorated with `@pytest.fixture`, and any test that names it as a parameter receives its value. **Parametrisation** runs the same test over many inputs without repetition, via `@pytest.mark.parametrize`.

In [ ]:
%%writefile test_advanced.py
import pytest

class ShoppingCart:
    def __init__(self):
        self.items = []
    def add(self, item, price):
        self.items.append((item, price))
    def total(self):
        return sum(price for _, price in self.items)

@pytest.fixture
def cart():
    # The fixture builds a cart; tests that ask for 'cart' receive a fresh one.
    c = ShoppingCart()
    c.add("book", 100)
    c.add("pen", 20)
    return c

def test_total(cart):
    assert cart.total() == 120

def test_add_increases_total(cart):
    cart.add("bag", 80)
    assert cart.total() == 200

# Parametrisation: one test, many input/expected pairs.
@pytest.mark.parametrize("value, expected", [
    (2, 4),
    (3, 9),
    (5, 25),
])
def test_square(value, expected):
    assert value ** 2 == expected

In [ ]:
!python -m pytest test_advanced.py -q

## Part 5 · Measuring Coverage

**Coverage** measures which lines of your code your tests actually run. It does not prove the tests are good, but it reveals code that no test touches at all, which is often where bugs hide. The `coverage` tool runs your tests while recording executed lines, then reports the percentage and the missing lines.

In [ ]:
%%writefile calculator.py
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        raise ValueError("cannot divide by zero")   # this branch may be untested
    return a / b

In [ ]:
%%writefile test_calculator.py
from calculator import add, subtract, multiply

# Note: we deliberately do not test divide(), so coverage will flag it.
def test_add():
    assert add(2, 3) == 5

def test_subtract():
    assert subtract(5, 2) == 3

def test_multiply():
    assert multiply(4, 3) == 12

In [ ]:
# Run the tests under coverage, then report which lines were missed.
!python -m coverage run -m pytest test_calculator.py -q
!python -m coverage report -m calculator.py

The report shows `calculator.py` is not fully covered: the `divide` function and its error branch were never executed, because no test calls them. This is exactly the signal coverage exists to give, pointing you to untested code.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A basic unittest case (Easy)

In [ ]:
import unittest

def reverse(text):
    return text[::-1]

class TestReverse(unittest.TestCase):
    def test_word(self):
        self.assertEqual(reverse("abc"), "cba")
    def test_empty(self):
        self.assertEqual(reverse(""), "")

unittest.main(argv=["ignored"], exit=False, verbosity=2)

### Example 2 — Testing for an exception (Easy)

In [ ]:
import unittest

def withdraw(balance, amount):
    if amount > balance:
        raise ValueError("insufficient funds")
    return balance - amount

class TestWithdraw(unittest.TestCase):
    def test_ok(self):
        self.assertEqual(withdraw(100, 30), 70)
    def test_too_much(self):
        with self.assertRaises(ValueError):
            withdraw(100, 200)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

### Example 3 — Using setUp for shared state (Medium)

In [ ]:
import unittest

class Counter:
    def __init__(self):
        self.value = 0
    def increment(self, by=1):
        self.value += by

class TestCounter(unittest.TestCase):
    def setUp(self):
        self.counter = Counter()        # fresh counter per test

    def test_starts_at_zero(self):
        self.assertEqual(self.counter.value, 0)

    def test_increment(self):
        self.counter.increment(5)
        self.assertEqual(self.counter.value, 5)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

### Example 4 — Mocking a dependency (Medium)

In [ ]:
import unittest
from unittest.mock import Mock

class OrderService:
    def __init__(self, payment_gateway):
        self.gateway = payment_gateway
    def checkout(self, amount):
        if self.gateway.charge(amount):
            return "confirmed"
        return "declined"

class TestOrderService(unittest.TestCase):
    def test_confirmed(self):
        gateway = Mock()
        gateway.charge.return_value = True       # simulate a successful charge
        service = OrderService(gateway)
        self.assertEqual(service.checkout(100), "confirmed")
        gateway.charge.assert_called_once_with(100)

    def test_declined(self):
        gateway = Mock()
        gateway.charge.return_value = False
        service = OrderService(gateway)
        self.assertEqual(service.checkout(100), "declined")

unittest.main(argv=["ignored"], exit=False, verbosity=2)

### Example 5 — A pytest suite with a fixture (Difficult)

In [ ]:
%%writefile test_inventory.py
import pytest

class Inventory:
    def __init__(self):
        self.stock = {}
    def add(self, item, qty):
        self.stock[item] = self.stock.get(item, 0) + qty
    def remove(self, item, qty):
        if self.stock.get(item, 0) < qty:
            raise ValueError("not enough stock")
        self.stock[item] -= qty
    def count(self, item):
        return self.stock.get(item, 0)

@pytest.fixture
def inventory():
    inv = Inventory()
    inv.add("apple", 10)
    return inv

def test_add(inventory):
    inventory.add("apple", 5)
    assert inventory.count("apple") == 15

def test_remove(inventory):
    inventory.remove("apple", 4)
    assert inventory.count("apple") == 6

def test_remove_too_many(inventory):
    with pytest.raises(ValueError):
        inventory.remove("apple", 100)

In [ ]:
!python -m pytest test_inventory.py -q

### Example 6 — Parametrised tests for many cases (Difficult)

In [ ]:
%%writefile test_validation.py
import pytest

def is_valid_username(name):
    return name.isalnum() and 3 <= len(name) <= 12

@pytest.mark.parametrize("name, expected", [
    ("alice", True),
    ("ab", False),          # too short
    ("a" * 13, False),      # too long
    ("user_1", False),      # underscore is not alphanumeric
    ("Bob42", True),
])
def test_username(name, expected):
    assert is_valid_username(name) == expected

In [ ]:
!python -m pytest test_validation.py -q

---

## Exercises

For `unittest` and `mock`, write the test in a cell and run it with `unittest.main(argv=["ignored"], exit=False)`. For `pytest`, write a `test_*.py` file with `%%writefile` and run it with `!python -m pytest`.

**Exercise 1 (Easy).** Write a `unittest.TestCase` for a function `square(n)` with two tests: one checking `square(4) == 16` and one checking `square(0) == 0`.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write a `unittest` test that checks a function `parse_int(s)` raises `ValueError` on the input `"abc"`, using `assertRaises`.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a `unittest.TestCase` using `setUp` to create a list `[1, 2, 3]` before each test, with one test checking its length and another appending an item and checking the new length.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Using `Mock`, test a class `Notifier` whose `send(msg)` calls `self.channel.deliver(msg)` and returns `True` if delivery returns `True`. Configure the mock channel and assert it was called correctly.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a pytest file (with `%%writefile`) containing a fixture that provides a dictionary `{"a": 1, "b": 2}`, plus a parametrised test that checks several key/value pairs are present. Run it with pytest.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python -m pytest cell here


---

## Solutions

**Solution 1**

In [ ]:
import unittest

def square(n):
    return n * n

class TestSquare(unittest.TestCase):
    def test_four(self):
        self.assertEqual(square(4), 16)
    def test_zero(self):
        self.assertEqual(square(0), 0)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

**Solution 2**

In [ ]:
import unittest

def parse_int(s):
    return int(s)

class TestParse(unittest.TestCase):
    def test_bad_input(self):
        with self.assertRaises(ValueError):
            parse_int("abc")

unittest.main(argv=["ignored"], exit=False, verbosity=2)

**Solution 3**

In [ ]:
import unittest

class TestList(unittest.TestCase):
    def setUp(self):
        self.data = [1, 2, 3]
    def test_length(self):
        self.assertEqual(len(self.data), 3)
    def test_append(self):
        self.data.append(4)
        self.assertEqual(len(self.data), 4)

unittest.main(argv=["ignored"], exit=False, verbosity=2)

**Solution 4**

In [ ]:
import unittest
from unittest.mock import Mock

class Notifier:
    def __init__(self, channel):
        self.channel = channel
    def send(self, msg):
        return self.channel.deliver(msg)

class TestNotifier(unittest.TestCase):
    def test_send(self):
        channel = Mock()
        channel.deliver.return_value = True
        notifier = Notifier(channel)
        self.assertTrue(notifier.send("hello"))
        channel.deliver.assert_called_once_with("hello")

unittest.main(argv=["ignored"], exit=False, verbosity=2)

**Solution 5**

In [ ]:
%%writefile test_solution5.py
import pytest

@pytest.fixture
def data():
    return {"a": 1, "b": 2}

@pytest.mark.parametrize("key, value", [("a", 1), ("b", 2)])
def test_pairs(data, key, value):
    assert data[key] == value

In [ ]:
!python -m pytest test_solution5.py -q

---

## Summary and Key Points

- `unittest.TestCase` groups tests as `test_*` methods; assertion methods (`assertEqual`, `assertTrue`, `assertRaises`, `assertAlmostEqual`) state expectations and give clear failures. Run programmatically in a notebook with `unittest.main(argv=["ignored"], exit=False)`.
- `setUp` runs before each test and `tearDown` after, giving every test a fresh, independent environment.
- `unittest.mock` isolates the unit under test: `Mock` records calls and returns configured values, and `patch` temporarily replaces a real dependency.
- `pytest` needs no classes: any `test_*` function with plain `assert` statements works, with fixtures supplied via parameters and `@pytest.mark.parametrize` for many input cases.
- Coverage measures which lines your tests execute, revealing untested code; high coverage is necessary but not sufficient for good tests.

### Next module: 6.2 — Debugging & Profiling

The next module covers finding and fixing problems and measuring performance: the debugger and `breakpoint()`, structured logging, and profiling with `cProfile` and `timeit`.